# NB04 — YOLO11n Training (imgsz=1280)

> **Pipeline position:** NB03 (Preprocessing) → **NB04 (YOLO11n)** → NB05 (YOLO11s) → NB06 (RT-DETR)

## Context coming in from NB03

| Item | Value |
|---|---|
| Train images | 15,480 (8,690 original + 6,790 oversampled) |
| Val images | 1,086 |
| Train bboxes | 191,463 |
| Minority classes | `vest` (7.4%), `no-vest` (8.8%) |
| Dominant class | `no-hardhat` (50.8%) — SHWD structural skew |
| GPU | RTX 4060 Laptop, 8.6GB VRAM |
| imgsz strategy | 1280 for all models (training_config.json) |

## Monitoring strategy

Per-class AP cannot be extracted cheaply during training callbacks — doing so requires a full val pass per epoch which doubles runtime. Instead:

- **Every epoch**: aggregate mAP50, cls_loss, recall logged to `results.csv` automatically by Ultralytics
- **Epochs 1, 5, 10, 20**: manual checkpoint eval cell run against saved weights — per-class AP50 extracted
- **Epoch 20**: primary decision gate — ablation triggered or training continues
- **Post-training**: full per-class evaluation on val and test sets

In [1]:
# ============================================================
# 0. IMPORTS & CONFIG
# ============================================================
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from datetime import datetime

import torch
from ultralytics import YOLO
from ultralytics.utils.plotting import plot_results

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

PROJECT_ROOT = Path('../')
RUNS_DIR     = PROJECT_ROOT / 'runs' / 'detect'
DATA_YAML    = PROJECT_ROOT / 'data' / 'ppe_dataset.yaml'
HYP_YAML     = PROJECT_ROOT / 'data' / 'hyp_ppe.yaml'
CFG_JSON     = PROJECT_ROOT / 'data' / 'training_config.json'
RESULTS_DIR  = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

CLASS_NAMES  = ['hardhat', 'no-hardhat', 'vest', 'no-vest', 'person']
MINORITY_CLS = ['vest', 'no-vest']   # from NB03 — primary ablation targets

# Load per-model training config written by NB03
with open(CFG_JSON) as f:
    ALL_CONFIGS = json.load(f)
cfg = ALL_CONFIGS['yolo11n']

print('Configuration loaded:')
for k, v in cfg.items():
    print(f'  {k:<20}: {v}')

Configuration loaded:
  imgsz               : 1280
  batch               : 8
  nbs                 : 64
  half                : True
  cache               : False
  workers             : 4
  grad_checkpoint     : False
  name                : ppe_yolo11n_1280


In [2]:
# ============================================================
# 1. PRE-FLIGHT CHECKS
# ============================================================
print('=' * 55)
print('  PRE-FLIGHT')
print('=' * 55)

# --- GPU ---
assert torch.cuda.is_available(), 'CUDA not available — check driver'
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
vram_free = (torch.cuda.get_device_properties(0).total_memory
             - torch.cuda.memory_allocated(0)) / 1e9
print(f'  GPU       : {torch.cuda.get_device_name(0)}')
print(f'  VRAM      : {vram_gb:.1f} GB total  |  {vram_free:.1f} GB free')

# VRAM warning — batch=8 at 1280 peaks ~10.9GB (spills into shared)
# batch=6 keeps it within physical VRAM
SAFE_BATCH = cfg['batch']   # 8 — survivable but in shared RAM
if vram_free < 8.0:
    SAFE_BATCH = 6
    print(f'  ⚠️  Free VRAM < 8GB — reducing batch: 8 → 6')
    print(f'     Effective batch stays 64 via accumulate=11')
else:
    print(f'  Batch     : {SAFE_BATCH} (effective {cfg["nbs"]} via grad accumulation)')

# --- Files ---
for label, path in [('Dataset YAML', DATA_YAML),
                     ('Hyp YAML',     HYP_YAML),
                     ('Config JSON',  CFG_JSON)]:
    status = '✅' if path.exists() else '❌  MISSING'
    print(f'  {label:<14}: {status}  {path}')

# --- Dataset quick count ---
train_imgs = len(list(
    (PROJECT_ROOT / 'data' / 'processed' / 'images' / 'train').glob('*.*')
))
print(f'  Train imgs : {train_imgs:,}')

print('=' * 55)

  PRE-FLIGHT
  GPU       : NVIDIA GeForce RTX 4060 Laptop GPU
  VRAM      : 8.6 GB total  |  8.6 GB free
  Batch     : 8 (effective 64 via grad accumulation)
  Dataset YAML  : ✅  ..\data\ppe_dataset.yaml
  Hyp YAML      : ✅  ..\data\hyp_ppe.yaml
  Config JSON   : ✅  ..\data\training_config.json
  Train imgs : 15,480


In [3]:
# ============================================================
# 2. MONITORING CALLBACKS
# Attached to the trainer before model.train() is called.
#
# on_val_end  → runs after every validation pass
#               logs aggregate metrics + fires warnings
#               when cls_loss or recall cross alert thresholds
#
# Checkpoints for per-class AP are done in Section 4 manually
# — running model.val() inside a training callback doubles
# runtime and is not worth the overhead.
# ============================================================

# Alert thresholds
CLS_LOSS_ALERT_EPOCH5  = 2.0   # cls_loss should be dropping by epoch 5
CLS_LOSS_ALERT_EPOCH20 = 1.5   # if still above here at ep 20 → intervene
RECALL_ALERT_EPOCH20   = 0.50  # aggregate recall floor at ep 20
MAP50_GATE_EPOCH20     = 0.55  # primary decision gate

# Metric log — written to disk so checkpoint cells can read it
METRIC_LOG_PATH = RESULTS_DIR / 'nb04_epoch_log.csv'
epoch_log = []   # list of dicts, one per epoch

def on_val_end(validator):
    """
    Ultralytics passes the DetectionValidator object here, not the trainer.
    Metrics live in validator.metrics, epoch in validator.training_data or 
    we track it ourselves via a counter.
    """
    global epoch_log

    # Epoch tracking — validator doesn't carry epoch, we count calls
    epoch = len(epoch_log) + 1

    # Metrics are on validator.metrics
    try:
        metrics   = validator.metrics
        map50     = float(metrics.box.map50)
        map5095   = float(metrics.box.map)
        precision = float(metrics.box.mp)    # mean precision
        recall    = float(metrics.box.mr)    # mean recall
    except Exception:
        map50 = map5095 = precision = recall = 0.0

    # Loss not available from validator — read from results.csv instead
    box_loss = cls_loss = dfl_loss = 0.0
    try:
        results_csv = RUNS_DIR / cfg['name'] / 'results.csv'
        if results_csv.exists():
            df_tmp = pd.read_csv(results_csv)
            df_tmp.columns = df_tmp.columns.str.strip()
            if len(df_tmp) > 0:
                last = df_tmp.iloc[-1]
                box_loss = float(last.get('train/box_loss', 0))
                cls_loss = float(last.get('train/cls_loss', 0))
                dfl_loss = float(last.get('train/dfl_loss', 0))
    except Exception:
        pass

    row = {
        'epoch':     epoch,
        'map50':     map50,
        'map5095':   map5095,
        'precision': precision,
        'recall':    recall,
        'box_loss':  box_loss,
        'cls_loss':  cls_loss,
        'dfl_loss':  dfl_loss,
        'timestamp': datetime.now().isoformat(),
    }
    epoch_log.append(row)
    pd.DataFrame(epoch_log).to_csv(METRIC_LOG_PATH, index=False)

    # ---- Alerts ----
    alerts = []
    if epoch == 5 and cls_loss > CLS_LOSS_ALERT_EPOCH5:
        alerts.append(f'⚠️  cls_loss={cls_loss:.3f} > {CLS_LOSS_ALERT_EPOCH5} at ep5 '
                      f'— no-hardhat dominance may be suppressing minority classes')

    if epoch == 20:
        if map50 < MAP50_GATE_EPOCH20:
            alerts.append(f'🔴 mAP50={map50:.3f} < {MAP50_GATE_EPOCH20} at ep20 '
                          f'— RUN Section 4 checkpoint eval before continuing')
        if recall < RECALL_ALERT_EPOCH20:
            alerts.append(f'⚠️  recall={recall:.3f} < {RECALL_ALERT_EPOCH20} at ep20 '
                          f'— minority class recall likely near zero')
        if cls_loss > CLS_LOSS_ALERT_EPOCH20:
            alerts.append(f'⚠️  cls_loss={cls_loss:.3f} > {CLS_LOSS_ALERT_EPOCH20} at ep20 '
                          f'— consider fl_gamma ablation')

    if epoch >= 15 and len(epoch_log) >= 10:
        recent      = [r['map50'] for r in epoch_log[-10:]]
        improvement = max(recent) - min(recent)
        if improvement < 0.005:
            alerts.append(f'⚠️  mAP50 stagnant for 10 epochs (Δ={improvement:.4f}) '
                          f'— early stop may trigger soon')

    if epoch in {1, 5, 10, 20, 50}:
        print(f'\n  📍 CHECKPOINT ep{epoch}: '
              f'mAP50={map50:.3f}  recall={recall:.3f}  cls_loss={cls_loss:.3f}')
        print(f'     → Run Section 4 checkpoint eval for per-class AP50')

    for alert in alerts:
        print(f'\n  {alert}')


print('✅  Callbacks defined')
print(f'   Metric log  → {METRIC_LOG_PATH}')
print(f'   Alert gates → ep5 cls_loss>{CLS_LOSS_ALERT_EPOCH5}'
      f' | ep20 mAP50<{MAP50_GATE_EPOCH20} | ep20 recall<{RECALL_ALERT_EPOCH20}')

✅  Callbacks defined
   Metric log  → ..\results\nb04_epoch_log.csv
   Alert gates → ep5 cls_loss>2.0 | ep20 mAP50<0.55 | ep20 recall<0.5


In [4]:
# Clear old callback, attach fixed version
model = YOLO('yolo11n.pt')
model.add_callback('on_val_end', on_val_end)

# Then call model.train(...) as before

In [5]:
import torch
torch.cuda.empty_cache()
print(f"Free VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.1f}GB")

Free VRAM: 8.6GB


In [6]:
# ============================================================
# 3. TRAINING RUN
# ============================================================
# If restarting from a checkpoint (e.g. after OOM or crash):
#   model = YOLO('runs/detect/ppe_yolo11n_1280/weights/last.pt')
#   results = model.train(resume=True)
# ============================================================

model = YOLO('yolo11n.pt')

# Attach monitoring callback
model.add_callback('on_val_end', on_val_end)

print(f'Starting YOLO11n training at imgsz={cfg["imgsz"]}...')
print(f'Estimated time: ~17h (batch={SAFE_BATCH}, 100 epochs, RTX 4060 Laptop)')
print(f'Checkpoint evals scheduled at epochs: 1, 5, 10, 20, 50')
print(f'Metric log updating live at: {METRIC_LOG_PATH}')
print()

train_start = time.time()
results = model.train(
    data      = str(DATA_YAML),
    cfg       = str(HYP_YAML),
    imgsz     = 1280,
    batch     = 6,       # was 8
    nbs       = 64,      # accumulate = 64/6 ≈ 11 — effective batch unchanged
    half      = True,
    cache     = False,
    workers   = 4,
    epochs    = 100,
    patience  = 20,
    device    = 0,
    name      = 'ppe_yolo11n_1280',
    save      = True,
    save_period = 5,
    plots     = True,
    val       = True,
    verbose   = True,
)

Starting YOLO11n training at imgsz=1280...
Estimated time: ~17h (batch=8, 100 epochs, RTX 4060 Laptop)
Checkpoint evals scheduled at epochs: 1, 5, 10, 20, 50
Metric log updating live at: ..\results\nb04_epoch_log.csv

New https://pypi.org/project/ultralytics/8.4.30 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.50  Python-3.11.9 torch-2.5.1+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: task=detect, mode=train, model=yolo11n.pt, data=..\data\ppe_dataset.yaml, epochs=100, time=None, patience=20, batch=6, imgsz=1280, save=True, save_period=5, cache=False, device=0, workers=4, project=None, name=ppe_yolo11n_12804, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid

train: Scanning C:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\PPE_Compliance_detection\data\processed\labels\train.cache... 15480 images, 0 backgrounds, 0 corrupt: 100%|██████████| 15480/15480 [00:00<?, ?it/s]

train: WARNING  C:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\PPE_Compliance_detection\data\processed\images\train\css_construction-682-_jpg.rf.7161480fa0134c8ed4985baec0b325c8_os1.jpg: 1 duplicate labels removed
train: WARNING  C:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\PPE_Compliance_detection\data\processed\images\train\shwd_000002.jpg: corrupt JPEG restored and saved
train: WARNING  C:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\PPE_Compliance_detection\data\processed\images\train\shwd_000006.jpg: corrupt JPEG restored and saved
train: WARNING  C:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\PPE_Compliance_detection\data\processed\images\train\shwd_000007.jpg: corrupt JPEG restored and saved
train: WARNING  C:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\PPE_Compliance_detection\data\processed\images\train\shwd_000011.jpg: corrupt JPEG restored and saved
train: WARNING  C:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\PPE_Compliance_detection\data\processed

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Scanning C:\Users\nwagb\Desktop\SponsorshipGlobalTalentPrep\PPE_Compliance_detection\data\processed\labels\val.cache... 1086 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1086/1086 [00:00<?, ?it/s]

...

                   all       1086      14198      0.908      0.824      0.905      0.609

  ⚠️  mAP50 stagnant for 10 epochs (Δ=0.0022) — early stop may trigger soon

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    100/100      5.35G      1.199     0.7199      1.297         79       1280: 100%|██████████| 2580/2580 [07:55<00:00,  5.43it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 91/91 [00:12<00:00,  7.36it/s]


                   all       1086      14198      0.907      0.825      0.905      0.609

  ⚠️  mAP50 stagnant for 10 epochs (Δ=0.0018) — early stop may trigger soon

100 epochs completed in 15.454 hours.
Optimizer stripped from runs\detect\ppe_yolo11n_12804\weights\last.pt, 5.6MB
Optimizer stripped from runs\detect\ppe_yolo11n_12804\weights\best.pt, 5.6MB

Validating runs\detect\ppe_yolo11n_12804\weights\best.pt...
Ultralytics 8.3.50  Python-3.11.9 torch-2.5.1+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 238 layers, 2,583,127 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 91/91 [00:12<00:00,  7.34it/s]


                   all       1086      14198      0.908      0.823      0.905      0.609
               hardhat        438       1109       0.91      0.844      0.918       0.69
            no-hardhat        638      11186       0.94      0.895      0.958      0.528
                  vest        133        315      0.902      0.794      0.893      0.615
               no-vest        182        377      0.919      0.814      0.897      0.597
                person        343       1211       0.87       0.77      0.863      0.616

  ⚠️  mAP50 stagnant for 10 epochs (Δ=0.0017) — early stop may trigger soon
Speed: 0.5ms preprocess, 4.8ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs\detect\ppe_yolo11n_12804


In [7]:
# ============================================================
# 4. CHECKPOINT EVAL
# ─────────────────────────────────────────────────────────
# RUN THIS CELL MANUALLY AT: epochs 1, 5, 10, 20, 50, done
#
# How to use mid-training:
#   1. Let training run in a terminal / background kernel
#   2. Open a second kernel on this same notebook
#   3. Set CHECKPOINT_EPOCH to whichever epoch you want
#   4. Run this cell — it loads that epoch's saved weights
#      and runs a full val pass for per-class AP50
#
# After training completes: set CHECKPOINT_EPOCH = 'best'
# ============================================================

CHECKPOINT_EPOCH = 'best'   # set to: 1 | 5 | 10 | 20 | 50 | 'best' | 'last'
RUN_NAME         = cfg['name']    # 'ppe_yolo11n_1280'

# Resolve weights path
if CHECKPOINT_EPOCH in ('best', 'last'):
    weights_path = RUNS_DIR / RUN_NAME / 'weights' / f'{CHECKPOINT_EPOCH}.pt'
else:
    # save_period=5 saves epoch5.pt, epoch10.pt, etc.
    weights_path = RUNS_DIR / RUN_NAME / 'weights' / f'epoch{CHECKPOINT_EPOCH}.pt'

assert weights_path.exists(), (
    f'Weights not found: {weights_path}\n'
    f'  For mid-training evals, checkpoint must have been saved '
    f'(save_period=5, so ep5/10/15... are available)'
)

print(f'Loading checkpoint: {weights_path}')
ckpt_model = YOLO(str(weights_path))

val_metrics = ckpt_model.val(
    data   = str(DATA_YAML),
    imgsz  = cfg['imgsz'],
    half   = cfg['half'],
    device = 0,
    split  = 'val',
    verbose= False,
)

# ---- Per-class AP50 table ----
ap50_per_class = val_metrics.box.ap50      # numpy array, one value per class
ap_per_class   = val_metrics.box.ap        # mAP50-95 per class
r_per_class    = val_metrics.box.r         # recall per class
p_per_class    = val_metrics.box.p         # precision per class

print(f'\n{"+" + "="*62 + "+"}')
print(f'  Checkpoint eval — epoch {CHECKPOINT_EPOCH} — {RUN_NAME}')
print(f'{"+" + "="*62 + "+"}')
print(f'  {"Class":<14} {"AP50":>8} {"AP50-95":>10} {"Precision":>11} {"Recall":>9}')
print(f'  {"-"*56}')

for i, cls_name in enumerate(CLASS_NAMES):
    ap50 = ap50_per_class[i] if i < len(ap50_per_class) else 0.0
    ap   = ap_per_class[i]   if i < len(ap_per_class)   else 0.0
    r    = r_per_class[i]    if i < len(r_per_class)    else 0.0
    p    = p_per_class[i]    if i < len(p_per_class)    else 0.0
    flag = '  ← MINORITY' if cls_name in MINORITY_CLS else ''
    warn = '  ⚠️ LOW'    if cls_name in MINORITY_CLS and ap50 < 0.40 else ''
    print(f'  {cls_name:<14} {ap50:>8.3f} {ap:>10.3f} {p:>11.3f} {r:>9.3f}{flag}{warn}')

print(f'  {"-"*56}')
map50_agg = float(val_metrics.box.map50)
map_agg   = float(val_metrics.box.map)
print(f'  {"ALL":<14} {map50_agg:>8.3f} {map_agg:>10.3f}')
print(f'{"+" + "="*62 + "+"}')

# ---- Ablation decision logic ----
vest_ap50    = ap50_per_class[2] if len(ap50_per_class) > 2 else 0.0
novest_ap50  = ap50_per_class[3] if len(ap50_per_class) > 3 else 0.0
minority_avg = (vest_ap50 + novest_ap50) / 2

print(f'\n  Minority class average AP50: {minority_avg:.3f}')

if CHECKPOINT_EPOCH in (20, 'best') or str(CHECKPOINT_EPOCH) == '20':
    print()
    print('  DECISION GATE (epoch 20 / best):')
    if minority_avg >= 0.50:
        print('  ✅  minority AP50 ≥ 0.50 — no ablation needed')
        print('     Proceed to NB05 (YOLO11s) with current hyp_ppe.yaml')
    elif 0.30 <= minority_avg < 0.50:
        print('  ⚠️  minority AP50 0.30–0.50 — marginal')
        print('     Recommended: run Section 5 (copy_paste ablation) before NB05')
    else:
        print('  🔴 minority AP50 < 0.30 — intervention required')
        print('     Run Section 5 (focal loss ablation) before NB05')

# Save checkpoint eval results
ckpt_result = {
    'checkpoint': str(CHECKPOINT_EPOCH),
    'map50':      map50_agg,
    'map5095':    map_agg,
}
for i, cls_name in enumerate(CLASS_NAMES):
    ckpt_result[f'ap50_{cls_name}'] = float(ap50_per_class[i]) if i < len(ap50_per_class) else 0.0

ckpt_log_path = RESULTS_DIR / 'nb04_checkpoint_evals.csv'
new_row = pd.DataFrame([ckpt_result])
if ckpt_log_path.exists():
    existing = pd.read_csv(ckpt_log_path)
    pd.concat([existing, new_row], ignore_index=True).to_csv(ckpt_log_path, index=False)
else:
    new_row.to_csv(ckpt_log_path, index=False)

print(f'\n  Saved to {ckpt_log_path}')

AssertionError: Weights not found: ..\runs\detect\ppe_yolo11n_1280\weights\best.pt
  For mid-training evals, checkpoint must have been saved (save_period=5, so ep5/10/15... are available)

In [8]:
# ============================================================
# 5. TRAINING CURVE MONITOR
# Run anytime during or after training to visualise progress.
# Reads results.csv written by Ultralytics + our epoch log.
# ============================================================

RUN_NAME = cfg['name']

# Ultralytics writes results.csv to the run directory
results_csv = RUNS_DIR / RUN_NAME / 'results.csv'
assert results_csv.exists(), f'results.csv not found at {results_csv} — has training started?'

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()   # Ultralytics adds leading spaces

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Training Monitor — {RUN_NAME}\n'
             f'Epoch {df.shape[0]} of 100  |  '
             f'Best mAP50 = {df["metrics/mAP50(B)"].max():.3f} '
             f'@ ep{df["metrics/mAP50(B)"].idxmax() + 1}',
             fontsize=14, fontweight='bold')

plot_configs = [
    # (col_name, ax, title, ylabel, alert_line)
    ('train/cls_loss',          axes[0,0], 'Classification Loss',     'cls_loss',  1.5),
    ('train/box_loss',          axes[0,1], 'Box Regression Loss',     'box_loss',  None),
    ('train/dfl_loss',          axes[0,2], 'DFL Loss',                'dfl_loss',  None),
    ('metrics/mAP50(B)',        axes[1,0], 'mAP50 (val)',             'mAP50',     0.55),
    ('metrics/recall(B)',       axes[1,1], 'Recall (val)',            'Recall',    0.50),
    ('metrics/precision(B)',    axes[1,2], 'Precision (val)',         'Precision', None),
]

epochs = df.index + 1
for col, ax, title, ylabel, alert in plot_configs:
    if col not in df.columns:
        ax.set_title(f'{title}\n(not in results.csv)')
        continue
    ax.plot(epochs, df[col], linewidth=2, color='steelblue')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))

    # Alert threshold line
    if alert is not None:
        ax.axhline(alert, color='red', linestyle='--', alpha=0.7,
                   label=f'Alert threshold ({alert})')
        ax.legend(fontsize=9)

    # Mark checkpoint epochs
    for ckpt_ep in [5, 10, 20, 50]:
        if ckpt_ep <= len(epochs):
            ax.axvline(ckpt_ep, color='orange', linestyle=':', alpha=0.5, linewidth=1)

plt.tight_layout()
plot_path = RESULTS_DIR / f'nb04_training_curves_ep{df.shape[0]}.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {plot_path}')

# Quick summary
latest = df.iloc[-1]
print(f'\nLatest epoch ({len(df)}):')
print(f'  mAP50     : {latest.get("metrics/mAP50(B)", 0):.4f}')
print(f'  Recall    : {latest.get("metrics/recall(B)", 0):.4f}')
print(f'  cls_loss  : {latest.get("train/cls_loss", 0):.4f}')

AssertionError: results.csv not found at ..\runs\detect\ppe_yolo11n_1280\results.csv — has training started?

In [ ]:
# ============================================================
# 6. TARGETED ABLATION — run only if Section 4 gate fires
# ─────────────────────────────────────────────────────────
# DO NOT RUN until you have epoch-20 checkpoint eval results.
# Choose ONE fix based on what the failure mode actually is:
#
#  SCENARIO A: vest/no-vest recall low, precision OK
#    → copy_paste 0.15 → 0.35  (more synthetic minority examples)
#
#  SCENARIO B: vest/no-vest both low (recall + precision)
#    → fl_gamma = 1.5  (focal loss penalises easy no-hardhat more)
#
#  SCENARIO C: cls_loss still > 1.5 at epoch 20
#    → cls loss weight 0.5 → 1.5  (harder classification boundary)
#
# Run the ablation variant for 30 epochs (not 100) — enough
# to see if minority AP50 improves vs baseline at same epoch.
# If it does, carry the winning config to NB05/NB06.
# ============================================================

ABLATION_SCENARIO = None   # set to: 'A' | 'B' | 'C'  when needed

if ABLATION_SCENARIO is None:
    print('Ablation not triggered. Run Section 4 checkpoint eval first.')
    print('Set ABLATION_SCENARIO = A / B / C based on the decision gate output.')

else:
    import yaml

    with open(HYP_YAML) as f:
        hyp = yaml.safe_load(f)

    ablation_name = f'ppe_yolo11n_1280_ablation{ABLATION_SCENARIO}'

    if ABLATION_SCENARIO == 'A':
        # Recall low, precision ok → more minority synthetic examples
        hyp['copy_paste'] = 0.35    # was 0.15
        print(f'Scenario A: copy_paste 0.15 → 0.35')

    elif ABLATION_SCENARIO == 'B':
        # Both metrics low → focal loss down-weights easy majority class
        hyp['fl_gamma'] = 1.5       # default is 0.0 (disabled)
        print(f'Scenario B: fl_gamma 0.0 → 1.5')

    elif ABLATION_SCENARIO == 'C':
        # cls_loss persistently high → harder classification boundary
        hyp['cls'] = 1.5            # was 0.5
        print(f'Scenario C: cls loss weight 0.5 → 1.5')

    # Write ablation hyp file
    ablation_hyp_path = PROJECT_ROOT / 'data' / f'hyp_ablation{ABLATION_SCENARIO}.yaml'
    with open(ablation_hyp_path, 'w') as f:
        yaml.dump(hyp, f, default_flow_style=False)
    print(f'Ablation hyp written → {ablation_hyp_path}')

    # Run ablation — 30 epochs only, compare at same epoch as baseline
    ablation_model = YOLO('yolo11n.pt')
    ablation_model.train(
        data       = str(DATA_YAML),
        cfg        = str(ablation_hyp_path),
        imgsz      = cfg['imgsz'],
        batch      = SAFE_BATCH,
        nbs        = cfg['nbs'],
        half       = cfg['half'],
        cache      = False,
        workers    = cfg['workers'],
        epochs     = 30,
        patience   = 15,
        device     = 0,
        name       = ablation_name,
        save       = True,
        save_period= 10,
        plots      = True,
        val        = True,
    )

    print(f'\nAblation {ABLATION_SCENARIO} complete.')
    print(f'Now re-run Section 4 with:')
    print(f'  CHECKPOINT_EPOCH = "best"')
    print(f'  RUN_NAME         = "{ablation_name}"')
    print(f'Compare minority AP50 vs baseline. If improved → update hyp_ppe.yaml')
    print(f'and carry winning config to NB05.')

In [9]:
# ============================================================
# 7. POST-TRAINING FINAL EVALUATION
# Run once training is fully complete.
# Evaluates best.pt on both val and test splits.
# Saves all metrics for NB07 comparison table.
# ============================================================

RUN_NAME     = cfg['name']
best_weights = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'

assert best_weights.exists(), f'best.pt not found — training must be complete'

final_model = YOLO(str(best_weights))

print('Running final evaluation on val and test splits...')
final_results = {}

for split in ('val', 'test'):
    m = final_model.val(
        data    = str(DATA_YAML),
        imgsz   = cfg['imgsz'],
        half    = cfg['half'],
        device  = 0,
        split   = split,
        verbose = False,
        save_json = True,
    )
    final_results[split] = m

    print(f'\n  {split.upper()} SET — best.pt')
    print(f'  {"-"*50}')
    print(f'  {"Class":<14} {"AP50":>8} {"AP50-95":>10} {"P":>8} {"R":>8}')
    print(f'  {"-"*50}')
    for i, cls_name in enumerate(CLASS_NAMES):
        ap50 = float(m.box.ap50[i]) if i < len(m.box.ap50) else 0.0
        ap   = float(m.box.ap[i])   if i < len(m.box.ap)   else 0.0
        p    = float(m.box.p[i])    if i < len(m.box.p)    else 0.0
        r    = float(m.box.r[i])    if i < len(m.box.r)    else 0.0
        flag = ' ←' if cls_name in MINORITY_CLS else ''
        print(f'  {cls_name:<14} {ap50:>8.3f} {ap:>10.3f} {p:>8.3f} {r:>8.3f}{flag}')
    print(f'  {"-"*50}')
    print(f'  {"ALL":<14} {float(m.box.map50):>8.3f} {float(m.box.map):>10.3f}')

# Persist for NB07 comparison table
save_record = {'model': RUN_NAME, 'imgsz': cfg['imgsz']}
for split in ('val', 'test'):
    m = final_results[split]
    save_record[f'{split}_map50']   = float(m.box.map50)
    save_record[f'{split}_map5095'] = float(m.box.map)
    for i, cls_name in enumerate(CLASS_NAMES):
        save_record[f'{split}_ap50_{cls_name}'] = (
            float(m.box.ap50[i]) if i < len(m.box.ap50) else 0.0
        )

nb07_log = RESULTS_DIR / 'nb07_model_comparison.csv'
new_row  = pd.DataFrame([save_record])
if nb07_log.exists():
    existing = pd.read_csv(nb07_log)
    # Replace existing row for this model if re-running
    existing = existing[existing['model'] != RUN_NAME]
    pd.concat([existing, new_row], ignore_index=True).to_csv(nb07_log, index=False)
else:
    new_row.to_csv(nb07_log, index=False)

print(f'\n✅  Final results saved → {nb07_log}')
print(f'    NB05 and NB06 will append to this same file.')
print(f'    NB07 loads it directly for the comparison table.')

AssertionError: best.pt not found — training must be complete

---
## Handoff to NB05

Before moving to NB05, confirm:

- [ ] Section 4 checkpoint eval run at epoch 20 and best
- [ ] Section 6 ablation decision made (run or skip)
- [ ] Section 7 complete — `nb07_model_comparison.csv` exists with YOLO11n row
- [ ] If ablation was run: `hyp_ppe.yaml` updated with winning config (copy across to NB05/NB06)

```python
# NB05 opening — update training_config.json if ablation changed hyp
cfg = ALL_CONFIGS['yolo11s']   # imgsz=1280, batch=4, nbs=64
model = YOLO('yolo11s.pt')
```